# 36 · openEO Studio and eodash: a reproducible handoff

**Time:** 50 minutes · **Audience:** undergraduate / introductory graduate GIS and Earth science.

**By the end:** generate an inspectable Studio recipe and an eodash configuration; explain credentials, processing and evidence.

[Run in JupyterLite](https://jltobias.github.io/JupyterLite-zarr-sql-views/lite/lab/index.html?path=36_eopf_studio_eodash.ipynb) · [Earth lab](https://jltobias.github.io/JupyterLite-zarr-sql-views/lab/) · [Sources and licenses](https://jltobias.github.io/JupyterLite-zarr-sql-views/book/credits.html)

Run cells from top to bottom with **Shift+Enter**. The first kernel start downloads Python WebAssembly packages.
The bundled exercises need no data-service account. Save a copy with **File → Save and Export Notebook As** before clearing browser storage.

![Four-stage workflow: STAC discovery, Zarr reading, SQL selection, linked views](assets/pipeline.svg)

In [1]:
from course import *
import numpy as np
import matplotlib.pyplot as plt
print('Ready: Python + NumPy + local teaching data')

Ready: Python + NumPy + local teaching data


**Reproducibility record.** Project: Earth in the Browser; author: course contributors;
email/organization: not supplied. Created/updated: 2026-11-21. Input: `data/eopf/`;
outputs: the figures and objects below (exports are named explicitly). Python, NumPy and
Matplotlib; SQLite where used. Initial EOPF teaching release. The acquisition is a
convenience example, not a representative sample. Missing values remain missing.

![EOPF workflow from discovery to direct maps, quality-aware analysis and server processing](assets/eopf-workflow.svg)
## 1. Build a recipe before opening Studio

[openEO Studio](https://studio.explorer.eopf.copernicus.eu/) is a browser editor for
Python that creates process graphs for a remote backend. Its
[script contract](https://studio.explorer.eopf.copernicus.eu/agents/script-contract.md)
provides `datacube`, `reduced` and `add_graph_to_map`; these names are not standard
standalone Python globals. `reduced` selects the first valid pixel, not a temporal
mean. The backend supplies physical reflectance, so applying DN scaling again is wrong.

We generate a small NDVI recipe and a prefill link. This does **not** submit a job or
save to an account. Studio requires login, then review and Save to execute/persist
the recipe. That external authenticated step is an optional instructor activity.

In [2]:
from eopf import studio_sample, studio_link, DATA, SITE
sample=studio_sample()
backend=json.loads((DATA/'openeo-collection.json').read_text())
available=backend['cube:dimensions']['bands']['values']
assert all(b in available for b in sample['bands'])
assert backend['cube:dimensions']['bands']['type']=='bands'
print(json.dumps(sample,indent=2))
# Export locally for inspection; no server job is created.
Path('eopf-studio-sample.json').write_text(json.dumps(sample,indent=2),encoding='utf-8')
display(HTML(f'<a href="{studio_link(sample)}" target="_blank" rel="noopener">Open reviewed recipe in Studio (login required)</a>'))

{
  "name": "EOPF teaching: southern Italy NDVI",
  "collectionId": "sentinel-2-l2a",
  "temporalRange": [
    "2026-09-22",
    "2026-09-29"
  ],
  "bands": [
    "reflectance|bands=b04",
    "reflectance|bands=b08"
  ],
  "boundingBox": [
    13.9,
    41.0,
    14.2,
    41.2
  ],
  "cloudCover": 20,
  "code": "# Physical reflectances: do not apply DN scaling a second time.\ndef ndvi(data):\n    red, nir = data[0], data[1]\n    total = nir + red\n    value = if_(total > 0, (nir - red) / total, None)\n    return array_create([value, value, value])\n\nindex = reduced.apply_dimension(dimension=\"bands\", process=ndvi)\ngrey = index.linear_scale_range(input_min=-1, input_max=1, output_min=0, output_max=255)\nadd_graph_to_map(grey.save_result(\"PNG\"), \"NDVI: first-valid composite\")\n"
}


## 2. Verify the process before interpreting the map

Inspect the date interval, bbox, ordered band names and first-valid reduction.
The sample is a grayscale NDVI display recipe with a denominator guard; it does not
reproduce notebook 32's SCL-fraction mask. A scene-level cloud filter is not that mask.
Studio graph generation and server execution are separate verification steps.
This course validates recipe fields against recorded backend metadata and Python
syntax; it does not claim to have run your authenticated project.

For an African port or agricultural area, first repeat lesson 31's coverage search.
Then alter bbox, dates and the research question. Sea-level rise needs projections
and vertical datums; water safety needs calibrated retrievals and field samples;
food security needs much more than an NDVI snapshot.

In [3]:
# Syntax validation is useful but is not remote process execution.
compile(sample['code'],'studio-recipe','exec')
print('Recipe syntax and ordered backend band names checked.')
print('Area:',sample['boundingBox'],'| dates:',sample['temporalRange'])

Recipe syntax and ordered backend band names checked.
Area: [13.9, 41.0, 14.2, 41.2] | dates: ['2026-09-22', '2026-09-29']


## 3. Use eodash to coordinate a catalog, map and controls

The [eodash integration](https://explorer.eopf.copernicus.eu/software-services/eodash/)
uses a shared state store and configuration for linked views. Our
[eodash page](https://jltobias.github.io/JupyterLite-zarr-sql-views/eopf-dashboard.html)
loads the actual pinned web component **on button click** and points it at the EOPF
STAC API. It is an external-service explorer, not an offline replica of the course.
The configuration below is the same one used by that page.

In [4]:
config=json.loads((DATA/'eodash-config.json').read_text())
assert config['stacEndpoint']['endpoint'].endswith('/stac')
assert config['stacEndpoint']['api'] is True
print(json.dumps(config,indent=2))
Path('eopf-dashboard-config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
display(HTML(f'<a href="{SITE}/eopf-dashboard.html" target="_blank" rel="noopener">Open actual eodash component</a>'))

{
  "id": "earth-browser-eopf",
  "stacEndpoint": {
    "endpoint": "https://api.explorer.eopf.copernicus.eu/stac",
    "api": true
  },
  "brand": {
    "noLayout": true,
    "name": "EOPF teaching explorer",
    "theme": {
      "colors": {
        "primary": "#123e48",
        "surface": "#ffffff"
      }
    }
  },
  "template": {
    "background": {
      "id": "map",
      "type": "internal",
      "widget": {
        "name": "EodashMap",
        "properties": {
          "center": [
            14.08,
            41.11
          ],
          "zoom": 10
        }
      }
    },
    "widgets": [
      {
        "id": "catalog",
        "type": "internal",
        "title": "Catalog and filters",
        "layout": {
          "x": 0,
          "y": 0,
          "w": 3,
          "h": 12
        },
        "widget": {
          "name": "EodashItemCatalog",
          "properties": {
            "datetimeFilter": true,
            "searchLimit": 10
          }
        }
      },
      

## 4. Publish an evidence-led story

Choose an item in eodash, adjust its date/filter controls and compare the visible
product ID with STAC Browser. A blank view may mean coverage, filter, service or
rendering failure. Record which one you established. In your evidence journal,
include item ID, acquisition date, CRS, resolution, quality rule, processing location,
rendering recipe and source credits. Connect to the existing AR6, FTW and SeasFire
studios only after aligning what each dataset can support.

## Try it yourself

Design a two-date follow-up for Lagos or Abidjan. Specify coverage checks, masks and additional evidence before changing any climate-impact claim.

## Check your reasoning

Two dates can show a difference, but seasonal timing, cloud/shadow errors, alignment and land management may explain it. A climate attribution requires a substantially stronger design.

## Evidence journal

Write four sentences: **what I observed; how I calculated it; what this cannot establish; what evidence I need next**.
For any figure you reuse, retain its units, date or scenario, data origin, transformations, and attribution.

## Sources and credit

- [EOPF Sentinel Zarr Explorer](https://explorer.eopf.copernicus.eu/), funded by ESA through the Copernicus Space Component programme; EOX, Development Seed and project contributors.
- [Copernicus Sentinel data legal notice](https://sentinels.copernicus.eu/documents/247904/690755/Sentinel_Data_Legal_Notice): contains modified Copernicus Sentinel data (2026). EOPF processing; course subset and repackaging.
- [Recorded source objects, checksums and transformations](https://jltobias.github.io/JupyterLite-zarr-sql-views/data/eopf/provenance.json). One 128 × 128 window of the upstream 120 m averaged overview, acquired 2026-09-28.
- [Studio prefill documentation](https://studio.explorer.eopf.copernicus.eu/agents/prefill.md) and [Studio source](https://github.com/developmentseed/openeo-studio), Development Seed, MIT.
- [eodash source](https://github.com/eodash/eodash), EOX and contributors, MIT.

Teaching adaptation of [dzole0311/zarr-sql-views](https://github.com/dzole0311/zarr-sql-views), ISC.
Original teaching code and prose: JupyterLite-zarr-sql-views contributors, ISC. Data keep their separate licenses.

In [5]:
print('LESSON COMPLETE: 36_eopf_studio_eodash')

LESSON COMPLETE: 36_eopf_studio_eodash
